In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("IncidentsStreamingViz")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark


: 

In [ ]:
from pyspark.sql.functions import col, from_json, expr
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

incident_schema = StructType([
    StructField("id",           StringType(), True),
    StructField("reported_at",  StringType(), True),
    StructField("lat",          DoubleType(), True),
    StructField("lon",          DoubleType(), True),
    StructField("alert_code",   StringType(), True),
    StructField("description",  StringType(), True),
    StructField("tag",          StringType(), True),
])

kafka_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "incidents_stream")
    .option("startingOffsets", "latest")
    .load()
)

json_df = kafka_df.selectExpr("CAST(value AS STRING) as json_str")

parsed_df = (
    json_df
    .select(from_json(col("json_str"), incident_schema).alias("data"))
    .select("data.*")
)

# Convertim reported_at (string) într-un timestamp Spark
incidents = parsed_df.withColumn(
    "reported_at_ts",
    expr("to_timestamp(reported_at)")
)

incidents.printSchema()


In [ ]:
from pyspark.sql.functions import window

agg_by_type = (
    incidents
    .groupBy(
        window(col("reported_at_ts"), "1 minute"),
        col("alert_code")
    )
    .count()
)

query = (
    agg_by_type
    .writeStream
    .outputMode("complete")
    .format("memory")             # scriem într-un „tabel” Spark in-memory
    .queryName("incidents_by_type")
    .start()
)


In [ ]:
import time
import pandas as pd

# Așteptăm un pic să ajungă date în stream
time.sleep(5)

pdf = spark.sql("""
    SELECT
        window.start AS window_start,
        window.end   AS window_end,
        alert_code,
        count
    FROM incidents_by_type
    ORDER BY window_start, alert_code
""").toPandas()

pdf.tail()


In [ ]:
import matplotlib.pyplot as plt

if pdf.empty:
    print("No data yet. Ensure producer + stream are running and try again.")
else:
    totals = (
        pdf.groupby("alert_code")["count"]
        .sum()
        .sort_values(ascending=False)
    )

    plt.figure(figsize=(6, 4))
    plt.bar(totals.index, totals.values)
    plt.xlabel("Alert code")
    plt.ylabel("Number of incidents")
    plt.title("Total incidents per alert_code")
    plt.tight_layout()
    plt.show()
